# T2 en GPU (CUDA / T4) — Evolución temporal densa del Liouvilliano con CuPy

**Entregable de Sebastián · capa HPC del efecto Mpemba cuántico.**

Este notebook lleva el *hotspot* de la evolución temporal de T2 —el producto de
matrices densas complejas $d\times d$ del Liouvilliano matrix-free— a la **GPU
T4** vía **CuPy** (que llama a **cuBLAS `zgemm`** por debajo). Es el tercer escalón
del escalado:

> **serie (1 hilo)  →  OpenMP (8 hilos, ~4×)  →  CUDA T4 (cuBLAS)**

Reproduce *exactamente* el mismo modelo de Ising disipativo que los binarios C++
(`rk4_evolution`), de modo que el observable $D_{HS}$ final calculado en la GPU se
puede **cruzar contra el valor de la CPU y contra el de tu RK4 en C++**
($D_{HS}\approx 0.3931849$ para $N=7$).

### Cómo ejecutarlo
1. `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.
2. `Entorno de ejecución → Ejecutar todo`.
3. Al final se descarga `resultados_t2_cuda.zip` (notebook ya ejecutado: usa
   `Archivo → Descargar → .ipynb`).


In [1]:
# --- 0. Comprobar GPU T4 y CuPy ---
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import numpy as np, cupy as cp, time, os, csv
print("NumPy :", np.__version__)
print("CuPy  :", cp.__version__)
print("GPU   :", cp.cuda.runtime.getDeviceProperties(0)['name'].decode())
os.makedirs("resultados_t2_cuda", exist_ok=True)
RES = "resultados_t2_cuda"


Tesla T4, 15360 MiB, 580.82.07
NumPy : 2.0.2
CuPy  : 14.0.1
GPU   : Tesla T4


## 1. Modelo y método (idénticos a `common/lindblad.hpp`)

Cadena de Ising transversa disipativa
$H=-J\sum Z_iZ_{i+1}-h\sum X_i$, canales locales
$L_{i,-}=\sqrt{\gamma(\bar n+1)}\,\sigma_i^-$, $L_{i,+}=\sqrt{\gamma\bar n}\,\sigma_i^+$.
Acción **matrix-free** del Liouvilliano
$\mathcal L[V]=-i(HV-VH)+\sum_\mu(L_\mu V L_\mu^\dagger-\tfrac12\{L_\mu^\dagger L_\mu,V\})$
e integrador **RK4**. Todo es *backend-agnóstico*: las mismas funciones corren en
NumPy (CPU) o CuPy (GPU) según el módulo `xp` de los arreglos.

In [2]:
# --- 1. Modelo (NumPy en CPU; se mueve a GPU con cp.asarray) ---
def n_bose(w, T):
    return 0.0 if w / T > 700 else 1.0 / (np.exp(w / T) - 1.0)

def build_ising(N, J, h, gamma, T):
    I2 = np.eye(2, dtype=complex)
    SX = np.array([[0,1],[1,0]], dtype=complex)
    SZ = np.array([[1,0],[0,-1]], dtype=complex)
    SM = np.array([[0,1],[0,0]], dtype=complex)   # bajada
    SP = np.array([[0,0],[1,0]], dtype=complex)   # subida
    def op_at(op, site):
        m = np.array([[1]], dtype=complex)
        for k in range(N):
            m = np.kron(m, op if k == site else I2)
        return m
    d = 2 ** N
    H = np.zeros((d, d), dtype=complex)
    for i in range(N):
        H += -h * op_at(SX, i)
    for i in range(N - 1):
        H += -J * (op_at(SZ, i) @ op_at(SZ, i + 1))
    nb = n_bose(2 * h if 2 * h > 0 else 1.0, T)
    Ls = []
    for i in range(N):
        Ls.append(np.sqrt(gamma * (nb + 1)) * op_at(SM, i))
        Ls.append(np.sqrt(gamma * nb)       * op_at(SP, i))
    return H, Ls, d

def gibbs_state(H, T, xp):
    d = H.shape[0]
    beta = 1.0 / T
    n = max(200, int(beta * 50)); dtau = beta / n
    G = xp.eye(d, dtype=complex)
    f = lambda X: -0.5 * (H @ X + X @ H)
    for _ in range(n):
        k1 = f(G); k2 = f(G + 0.5*dtau*k1); k3 = f(G + 0.5*dtau*k2); k4 = f(G + dtau*k3)
        G = G + dtau * (k1 + 2*k2 + 2*k3 + k4) / 6.0
    return G / xp.trace(G)

def precompute(Ls):
    Lds = [L.conj().T for L in Ls]
    LdL = [Lds[i] @ Ls[i] for i in range(len(Ls))]
    return Lds, LdL

def apply_lindblad(H, Ls, Lds, LdL, V):
    out = -1j * (H @ V - V @ H)
    for mu in range(len(Ls)):
        out = out + Ls[mu] @ V @ Lds[mu] - 0.5 * (LdL[mu] @ V + V @ LdL[mu])
    return out

def rk4_step(H, Ls, Lds, LdL, rho, dt):
    k1 = apply_lindblad(H, Ls, Lds, LdL, rho)
    k2 = apply_lindblad(H, Ls, Lds, LdL, rho + 0.5*dt*k1)
    k3 = apply_lindblad(H, Ls, Lds, LdL, rho + 0.5*dt*k2)
    k4 = apply_lindblad(H, Ls, Lds, LdL, rho + dt*k3)
    return rho + dt * (k1 + 2*k2 + 2*k3 + k4) / 6.0

def d_hs(rho, rss):
    diff = rho - rss
    return float(((diff @ diff).trace().real) ** 0.5)

def to_backend(H, Ls, xp):
    if xp is cp:
        return cp.asarray(H), [cp.asarray(L) for L in Ls]
    return H, Ls

def sync(xp):
    if xp is cp: cp.cuda.Device().synchronize()

def evolve_timed(H, Ls, rho0, nsteps, dt, xp):
    Lds, LdL = precompute(Ls)
    rho = rho0.copy()
    sync(xp); t0 = time.perf_counter()
    for _ in range(nsteps):
        rho = rk4_step(H, Ls, Lds, LdL, rho, dt)
    sync(xp); return rho, time.perf_counter() - t0

print("modelo listo.")


modelo listo.


## 2. Validación: GPU ↔ CPU ↔ C++

Se evoluciona la misma preparación (Gibbs a $T_0=5$) relajando al baño ($T=0.8$)
para $N=7$ y se comprueba que:
- el $D_{HS}$ final de **GPU y CPU coinciden a precisión de máquina** (la GPU no
  altera la física, igual rigor que el cross-check serie/paralelo);
- ambos coinciden con el **valor de referencia del RK4 en C++**, $0.3931849$.

In [3]:
# --- 2. Validación N=7 ---
N, J, h, gamma, Tbath, T0 = 7, 1.0, 0.5, 0.4, 0.8, 5.0
t_max, dt = 6.0, 0.005
nsteps = int(round(t_max / dt))
CPP_REF = 0.3931849204   # valor del binario rk4_evolution (config T2a)

H, Ls, d = build_ising(N, J, h, gamma, Tbath)

# CPU
rho0_c = gibbs_state(H, T0, np); rss_c = gibbs_state(H, Tbath, np)
rho_c, t_c = evolve_timed(H, Ls, rho0_c, nsteps, dt, np)
dhs_c = d_hs(rho_c, rss_c)

# GPU
Hg, Lsg = to_backend(H, Ls, cp)
rho0_g = gibbs_state(Hg, T0, cp); rss_g = gibbs_state(Hg, Tbath, cp)
rho_g, t_g = evolve_timed(Hg, Lsg, rho0_g, nsteps, dt, cp)
dhs_g = d_hs(rho_g, rss_g)

print(f"N={N}, d={d}, pasos={nsteps}")
print(f"  D_HS  CPU  = {dhs_c:.10f}   ({t_c:.2f}s)")
print(f"  D_HS  GPU  = {dhs_g:.10f}   ({t_g:.2f}s)   speedup={t_c/t_g:.1f}x")
print(f"  D_HS  C++  = {CPP_REF:.10f}  (referencia rk4_evolution)")
print(f"  |GPU - CPU| = {abs(dhs_g-dhs_c):.2e}")
print(f"  |GPU - C++| = {abs(dhs_g-CPP_REF):.2e}")
assert abs(dhs_g - dhs_c) < 1e-9,  "GPU y CPU difieren!"
assert abs(dhs_g - CPP_REF) < 1e-3, "no coincide con el RK4 en C++!"
print("  -> VALIDACION OK")


N=7, d=128, pasos=1200
  D_HS  CPU  = 0.3931849204   (154.70s)
  D_HS  GPU  = 0.3931849204   (27.03s)   speedup=5.7x
  D_HS  C++  = 0.3931849204  (referencia rk4_evolution)
  |GPU - CPU| = 0.00e+00
  |GPU - C++| = 2.64e-11
  -> VALIDACION OK


## 3. Benchmark A — Evolución RK4 completa: CPU vs GPU vs tamaño $N$

Tiempo de la evolución (nº de pasos fijo) en función de $N$ (malla $d=2^N$). Para
$d$ pequeño la GPU pierde por el coste de lanzar kernels; al crecer la malla, la
GPU domina. Es la tercera curva de tu figura *serie → OpenMP → GPU*.

In [4]:
# --- 3. Benchmark A: evolucion RK4 vs N ---
Ns = [4, 5, 6, 7, 8, 9]
steps_bench, dt_bench = 120, 0.01
rowsA = []
for Nb in Ns:
    Hb, Lsb, db = build_ising(Nb, J, h, gamma, Tbath)
    r0 = gibbs_state(Hb, T0, np)
    _, tcpu = evolve_timed(Hb, Lsb, r0, steps_bench, dt_bench, np)
    Hgb, Lsgb = to_backend(Hb, Lsb, cp)
    r0g = cp.asarray(r0)
    _, _ = evolve_timed(Hgb, Lsgb, r0g, 3, dt_bench, cp)   # warm-up
    _, tgpu = evolve_timed(Hgb, Lsgb, r0g, steps_bench, dt_bench, cp)
    rowsA.append((Nb, db, tcpu, tgpu, tcpu/tgpu))
    print(f"  N={Nb:2d} d={db:5d}  CPU={tcpu:8.3f}s  GPU={tgpu:7.3f}s  speedup={tcpu/tgpu:6.1f}x")

with open(f"{RES}/t2_cuda_evolution.csv","w",newline="") as f:
    w=csv.writer(f); w.writerow(["N","d","t_cpu_s","t_gpu_s","speedup"]); w.writerows(rowsA)

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
A=np.array(rowsA,dtype=float)
fig,ax=plt.subplots(1,2,figsize=(11,4.2))
ax[0].semilogy(A[:,0],A[:,2],"o-",color="C3",lw=1.9,label="CPU (NumPy/BLAS)")
ax[0].semilogy(A[:,0],A[:,3],"s-",color="C0",lw=1.9,label="GPU T4 (CuPy/cuBLAS)")
ax[0].set_xlabel("N (espines, malla $d=2^N$)"); ax[0].set_ylabel("tiempo de computo [s]")
ax[0].set_title("(a) Evolucion RK4: CPU vs GPU T4"); ax[0].legend(); ax[0].grid(alpha=.3,which="both"); ax[0].set_xticks(A[:,0])
ax[1].plot(A[:,0],A[:,4],"^-",color="C2",lw=1.9); ax[1].axhline(1,color="k",ls="--",alpha=.5)
ax[1].set_xlabel("N (espines)"); ax[1].set_ylabel("speedup $T_{CPU}/T_{GPU}$")
ax[1].set_title("(b) Aceleracion por GPU"); ax[1].grid(alpha=.3); ax[1].set_xticks(A[:,0])
fig.tight_layout(); fig.savefig(f"{RES}/fig_t2_cuda_evolution.png",dpi=140); plt.show()


  N= 4 d=   16  CPU=   0.101s  GPU=  1.001s  speedup=   0.1x
  N= 5 d=   32  CPU=   0.491s  GPU=  1.428s  speedup=   0.3x
  N= 6 d=   64  CPU=   1.771s  GPU=  2.268s  speedup=   0.8x
  N= 7 d=  128  CPU=  14.790s  GPU=  2.754s  speedup=   5.4x
  N= 8 d=  256  CPU= 115.279s  GPU= 22.940s  speedup=   5.0x
  N= 9 d=  512  CPU= 942.911s  GPU=189.596s  speedup=   5.0x


## 4. Benchmark B — El *hotspot* aislado: GEMM complejo $d\times d$ (cuBLAS `zgemm`)

El producto de matrices densas es el corazón de `apply_lindblad`. Aquí se mide en
aislado (CPU/BLAS vs GPU/cuBLAS) frente a $d$, hasta tamaños inviables para la
evolución completa. Es la curva más limpia del poder bruto de la T4.

In [5]:
# --- 4. Benchmark B: GEMM complejo d x d ---
ds = [64, 128, 256, 512, 1024, 2048]
reps = 5
rowsB = []
for dd in ds:
    A1 = (np.random.randn(dd,dd)+1j*np.random.randn(dd,dd)).astype(complex)
    B1 = (np.random.randn(dd,dd)+1j*np.random.randn(dd,dd)).astype(complex)
    t0=time.perf_counter()
    for _ in range(reps): C1=A1@B1
    tcpu=(time.perf_counter()-t0)/reps
    Ag=cp.asarray(A1); Bg=cp.asarray(B1)
    C2=Ag@Bg; cp.cuda.Device().synchronize()           # warm-up
    cp.cuda.Device().synchronize(); t0=time.perf_counter()
    for _ in range(reps): C2=Ag@Bg
    cp.cuda.Device().synchronize(); tgpu=(time.perf_counter()-t0)/reps
    gflops = 8.0*dd**3/1e9   # GEMM complejo ~ 8 d^3 FLOP
    rowsB.append((dd,tcpu,tgpu,tcpu/tgpu,gflops/tgpu))
    print(f"  d={dd:5d}  CPU={tcpu*1e3:8.2f}ms  GPU={tgpu*1e3:7.2f}ms  speedup={tcpu/tgpu:6.1f}x  GPU={gflops/tgpu:7.1f} GFLOP/s")

with open(f"{RES}/t2_cuda_gemm.csv","w",newline="") as f:
    w=csv.writer(f); w.writerow(["d","t_cpu_s","t_gpu_s","speedup","gpu_gflops"]); w.writerows(rowsB)

Bm=np.array(rowsB,dtype=float)
fig,ax=plt.subplots(1,2,figsize=(11,4.2))
ax[0].loglog(Bm[:,0],Bm[:,1],"o-",color="C3",lw=1.9,label="CPU (NumPy/BLAS)")
ax[0].loglog(Bm[:,0],Bm[:,2],"s-",color="C0",lw=1.9,label="GPU T4 (cuBLAS zgemm)")
ax[0].set_xlabel("d (dimension de la matriz)"); ax[0].set_ylabel("tiempo por GEMM [s]")
ax[0].set_title("(a) GEMM complejo $d\\times d$"); ax[0].legend(); ax[0].grid(alpha=.3,which="both")
ax[1].semilogx(Bm[:,0],Bm[:,3],"^-",color="C2",lw=1.9); ax[1].axhline(1,color="k",ls="--",alpha=.5)
ax[1].set_xlabel("d"); ax[1].set_ylabel("speedup $T_{CPU}/T_{GPU}$")
ax[1].set_title("(b) Aceleracion del hotspot"); ax[1].grid(alpha=.3,which="both")
fig.tight_layout(); fig.savefig(f"{RES}/fig_t2_cuda_gemm.png",dpi=140); plt.show()


  d=   64  CPU=    0.96ms  GPU=   0.10ms  speedup=   9.2x  GPU=   20.1 GFLOP/s
  d=  128  CPU=    0.39ms  GPU=   0.11ms  speedup=   3.7x  GPU=  158.4 GFLOP/s
  d=  256  CPU=    2.68ms  GPU=   0.73ms  speedup=   3.7x  GPU=  184.8 GFLOP/s
  d=  512  CPU=   21.35ms  GPU=   5.32ms  speedup=   4.0x  GPU=  201.8 GFLOP/s
  d= 1024  CPU=  171.36ms  GPU=  56.52ms  speedup=   3.0x  GPU=  152.0 GFLOP/s
  d= 2048  CPU= 1583.64ms  GPU= 274.73ms  speedup=   5.8x  GPU=  250.1 GFLOP/s


## 5. Guardar y descargar resultados

Empaqueta CSV + figuras. **Para la versión ejecutada del notebook**, usa además
`Archivo → Descargar → Descargar .ipynb`.

In [6]:
# --- 5. Empaquetar y descargar ---
import shutil
shutil.make_archive("resultados_t2_cuda","zip",RES)
print("Contenido:", os.listdir(RES))
try:
    from google.colab import files
    files.download("resultados_t2_cuda.zip")
except Exception as e:
    print("(descarga manual de resultados_t2_cuda.zip)", e)


Contenido: ['fig_t2_cuda_evolution.png', 'fig_t2_cuda_gemm.png', 't2_cuda_evolution.csv', 't2_cuda_gemm.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Lectura para la exposición

- **Validación cruzada triple**: el $D_{HS}$ de la **GPU** coincide con el de la
  **CPU** (precisión de máquina) y con el del **RK4 en C++** → la GPU no cambia la
  física, solo la acelera.
- **Benchmark A** añade el tercer escalón a tu figura *serie → OpenMP → CUDA*: la
  GPU pierde para $d$ pequeño (overhead de kernels) y gana al crecer la malla.
- **Benchmark B** aísla el *hotspot* (`zgemm`): la T4 muestra su poder bruto en el
  producto de matrices densas, el mismo que paraleliza OpenMP en la CPU.
- **Conclusión HPC**: el régimen denso de T2 es *compute/bandwidth-bound* y mapea
  de forma natural a cuBLAS; CUDA es el tercer paradigma (junto a OpenMP intranodo
  y MPI internodo) del cuadro §7.4 del informe maestro.


In [7]:
import shutil
import os
from google.colab import files

RES = "resultados_t2_cuda"

shutil.make_archive(RES, "zip", RES)
print("Contenido:", os.listdir(RES))

try:
    files.download(f"{RES}.zip")
except Exception as e:
    print(f"(descarga manual de {RES}.zip)", e)

Contenido: ['fig_t2_cuda_evolution.png', 'fig_t2_cuda_gemm.png', 't2_cuda_evolution.csv', 't2_cuda_gemm.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>